In [1]:
import pandas as pd
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

In [2]:
from pathlib import Path

BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"

In [3]:
df = pd.read_csv(f'{DATA_DIR}/v2/function_dataset_cleaned_v2.csv')

In [4]:
# 定義特徵與目標變數
X = df.drop(columns=['label'])
y = df['label']

In [5]:
# 將數據分割為訓練集與測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10, stratify=y)

In [6]:
print(y_train.value_counts())       # 訓練集類別

label
cubic          400
cosine         400
exponential    400
quadratic      400
reciprocal     400
sine           400
logarithmic    400
linear         400
Name: count, dtype: int64


In [7]:
print(y_test.value_counts())        # 測試集類別

label
quadratic      100
logarithmic    100
reciprocal     100
sine           100
cosine         100
exponential    100
cubic          100
linear         100
Name: count, dtype: int64


## 開始訓練 V2 模型

In [8]:
# 建立pipeline, 將數據標準化, 並蒐集模型
models = {"LogisticRegression": make_pipeline(StandardScaler(),
                                              LogisticRegression(max_iter=1000)),
          "DecisionTree": DecisionTreeClassifier(random_state=10),
          
          "RandomForest": RandomForestClassifier(random_state=10),
          
          "KNN": make_pipeline(StandardScaler(),
                               KNeighborsClassifier(n_neighbors=3)),
          "SVM": make_pipeline(StandardScaler(),
                               SVC())
         }

In [9]:
model_times = []
for name, model in models.items():
    start_time = time.perf_counter()        # 開始計時
    model.fit(X_train, y_train)
    end_time = time.perf_counter()          # 結束計時
    elapsed_time = end_time - start_time    # 計算經過時間
    model_times.append(elapsed_time)

    # 儲存到joblib
    joblib.dump(model, f'{MODEL_DIR}/v2/{name}_baseline_v2.joblib')

In [10]:
print(model_times)

[0.050827400002162904, 0.0465095000108704, 0.9358665000036126, 0.0059889999974984676, 0.2536015000077896]


## 預測模型

In [11]:
from sklearn.metrics import *

In [12]:
results = []
trained_models = {}
time_index = 0
for name, model in models.items():
    model = joblib.load(f'{MODEL_DIR}/v2/{name}_baseline_v2.joblib')
    trained_models[name] = model        # 儲存模型物件

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"正在預測 {name} 模型:")
    print(f"準確度: {acc:.4f}")
    print(f"混淆矩陣:\n{confusion_matrix(y_test, y_pred)}")
    print(f"分類報告:\n{classification_report(y_test, y_pred)}\n")
    results.append({'Model': name, 
                    'Accuracy': acc, 
                    'Precision': pre, 
                    'Recall': recall, 
                    'F1': f1,
                    'Time': model_times[time_index]})
    time_index += 1

正在預測 LogisticRegression 模型:
準確度: 0.1913
混淆矩陣:
[[13  4 17  6 31 15  1 13]
 [13  9 21 17 15 15  0 10]
 [ 8 11 21 24 18 14  0  4]
 [ 0  7  8 25 41  3  0 16]
 [ 0  1 28  7 43 16  1  4]
 [ 5 14 19 23 18 14  0  7]
 [ 0  3  4  2 45 41  5  0]
 [10  4 12 28 19  4  0 23]]
分類報告:
              precision    recall  f1-score   support

      cosine       0.27      0.13      0.17       100
       cubic       0.17      0.09      0.12       100
 exponential       0.16      0.21      0.18       100
      linear       0.19      0.25      0.22       100
 logarithmic       0.19      0.43      0.26       100
   quadratic       0.11      0.14      0.13       100
  reciprocal       0.71      0.05      0.09       100
        sine       0.30      0.23      0.26       100

    accuracy                           0.19       800
   macro avg       0.26      0.19      0.18       800
weighted avg       0.26      0.19      0.18       800


正在預測 DecisionTree 模型:
準確度: 0.5800
混淆矩陣:
[[56  0  1  4  8  5  9 17]
 [ 0 67 13  

In [13]:
results_df = pd.DataFrame(results)

In [14]:
# 儲存成網頁
results_df.to_html(f'{DATA_DIR}/v2/model_results_v2.html')

## 模型比較

In [15]:
print(results_df.to_string(index=False, formatters={'Accuracy': '{:.4f}'.format, 
                                                    'Precision': '{:.4f}'.format, 
                                                    'Recall': '{:.4f}'.format, 
                                                    'F1': '{:.4f}'.format,
                                                    'Time': '{:.4f}'.format}))

             Model Accuracy Precision Recall     F1   Time
LogisticRegression   0.1913    0.2626 0.1913 0.1788 0.0508
      DecisionTree   0.5800    0.5819 0.5800 0.5805 0.0465
      RandomForest   0.7087    0.7203 0.7087 0.7108 0.9359
               KNN   0.6012    0.6118 0.6012 0.5985 0.0060
               SVM   0.4700    0.5975 0.4700 0.4682 0.2536


### 找出 F1 最高的模型

In [16]:
best_result = results_df.loc[results_df['F1'].idxmax()]
best_model_name = best_result['Model']
best_model = trained_models[best_model_name]
print(f"最佳模型:\n{best_result}")

最佳模型:
Model        RandomForest
Accuracy          0.70875
Precision        0.720304
Recall            0.70875
F1               0.710817
Time             0.935867
Name: 2, dtype: object


In [17]:
joblib.dump(best_model, f'{MODEL_DIR}/v2/best_model_v2.joblib')

['d:\\Python\\我的AI作品集\\專案1_數學函數辨識\\Package\\models/v2/best_model_v2.joblib']